# Генератор данных к лабораторной работе №6
## Корреляционный анализ: признаки дробления бизнеса (вариант 1)

Скрипт собирает книгу Excel `ЛР6_данные.xlsx` с листом **Вариант 1**: 16 компаний × 36 месяцев (2023-01 … 2025-12) × 8 столбцов.

Столбцы: `Дата | Компания | Режим | Выручка_млн | Операционная_маржа_% | Управленческие_расходы_тыс | Численность_чел | Доля_крупнейшего_покупателя_%`

Порядок генерации:
1. ядро пула (3 компании);
2. активные и технические сателлиты (4 компании) из того же шокового канала;
3. зеркальная компания с высокой обратной корреляцией к «Северному Ветру»;
4. независимые ОСНО / УСН и пограничные-ловушки — итеративно, пока корреляции и доли от лимита попадают в целевые диапазоны;
5. проверка лимита УСН (сумма 12 месяцев ≤ 450 млн) и отчёт по всем маркерам.

В спецификации групп перечислены 16 компаний (включая «Прима-Групп»). Запустите все ячейки сверху вниз.


In [ ]:
"""Генератор данных ЛР6 «Дробление бизнеса», вариант 1."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

SEED = 20260101
USN_LIMIT = 450.0
USN_HC_LIMIT = 100
MAX_TRIES = 300
R_IND_MIN, R_IND_MAX = 0.25, 0.40
R_BORDER_MIN, R_BORDER_MAX = 0.20, 0.42
R_MIRROR_MAX = -0.70
R_SAT_MIN = 0.55
R_SAT_MAX = 0.85
OUTPUT_FILE = "ЛР6_данные.xlsx"
SHEET_NAME = "Вариант 1"

DATES = pd.date_range("2023-01-01", "2025-12-01", freq="MS")
YEARS = (2023, 2024, 2025)
N = len(DATES)


@dataclass
class Firm:
    name: str
    regime: str
    role: str
    margin_mean: float
    margin_std: float
    mgmt_share: float
    mgmt_flat: bool
    headcount: int
    top_buyer: float
    trend: float
    target_share: float | None = None
    base_rev: float | None = None
    mirror_of: str | None = None


POOL_CORE = [
    Firm("Северный Ветер", "УСН", "Ядро пула", 12, 1.5, 0.10, False, 40, 35, 0.10, target_share=0.94),
    Firm("Прима", "УСН", "Ядро пула", 11, 1.5, 0.10, False, 38, 40, 0.10, target_share=0.95),
    Firm("Гранит", "УСН", "Ядро пула", 12, 1.5, 0.10, False, 36, 38, 0.10, target_share=0.94),
]
POOL_SAT = [
    Firm("Ветер-Опт", "УСН", "Активный сателлит", 10, 2, 0.09, False, 20, 45, 0.09, target_share=0.86),
    Firm("Ветер-Логистик", "УСН", "Активный сателлит", 9, 2, 0.08, False, 18, 50, 0.09, target_share=0.84),
]
POOL_TECH = [
    Firm("Прима-Трейд", "УСН", "Технический сателлит", 15, 1, 0.02, True, 6, 85, 0.08, target_share=0.88),
    Firm("Гранит-Плюс", "УСН", "Технический сателлит", 15, 1, 0.02, True, 5, 82, 0.08, target_share=0.86),
]
MIRROR = [
    Firm("Ветер-Юг", "УСН", "Зеркальная компания", 10, 2, 0.09, False, 16, 55, 0.09, target_share=0.85, mirror_of="Северный Ветер"),
]
IND_OSNO = [
    Firm("Лагуна", "ОСНО", "Независимая (контроль)", 8, 3, 0.15, False, 110, 20, 0.05, base_rev=40.0),
    Firm("Меркурий", "ОСНО", "Независимая (контроль)", 7, 3, 0.16, False, 115, 22, 0.04, base_rev=42.0),
    Firm("Орион", "ОСНО", "Независимая (контроль)", 8, 3, 0.14, False, 108, 18, 0.05, base_rev=38.0),
]
IND_USN = [
    Firm("Лагуна-Строй", "УСН", "Независимая УСН", 9, 3, 0.20, False, 25, 25, 0.05, target_share=0.50),
    Firm("Меркурий-Юг", "УСН", "Независимая УСН", 9, 3, 0.22, False, 22, 28, 0.04, target_share=0.45),
]
IND_BORDER = [
    Firm("Ветер-Сервис", "УСН", "Пограничная-ловушка", 10, 3, 0.18, False, 15, 30, 0.06),
    Firm("Прима-Снаб", "УСН", "Пограничная-ловушка", 10, 3, 0.19, False, 14, 32, 0.06),
    Firm("Прима-Групп", "УСН", "Пограничная-ловушка", 9, 3, 0.17, False, 13, 35, 0.06),
]

CORE_NAMES = [f.name for f in POOL_CORE]
BRAKE_NAMES = {"Прима", "Гранит"}


def _seasonal(month: int) -> float:
    base = 1 + 0.15 * np.sin((month - 1) / 12 * 2 * np.pi - np.pi / 2)
    dec_boost = 1.12 if month == 12 else 1.0
    jan_drop = 0.85 if month == 1 else 1.0
    return base * dec_boost * jan_drop


def seasonal_vector() -> np.ndarray:
    return np.array([_seasonal(d.month) for d in DATES], dtype=float)


def standardize(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    std = x.std(ddof=0)
    if std < 1e-12:
        return np.zeros_like(x)
    return (x - x.mean()) / std


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def spearman(a: np.ndarray, b: np.ndarray) -> float:
    return pearson(pd.Series(a).rank().to_numpy(), pd.Series(b).rank().to_numpy())


def mean_r_with_core(series: np.ndarray, core: dict[str, np.ndarray]) -> float:
    return float(np.mean([pearson(series, core[name]) for name in CORE_NAMES]))


def yearly_sums(rev: np.ndarray) -> np.ndarray:
    return rev.reshape(3, 12).sum(axis=1)


def year_shares(rev: np.ndarray) -> np.ndarray:
    return yearly_sums(rev) / USN_LIMIT * 100


def chain_growth(rev: np.ndarray) -> np.ndarray:
    return np.diff(rev) / np.maximum(rev[:-1], 1e-9)


def ytd_share(rev: np.ndarray) -> np.ndarray:
    out = np.zeros_like(rev, dtype=float)
    for y in range(3):
        sl = slice(y * 12, (y + 1) * 12)
        out[sl] = np.cumsum(rev[sl]) / USN_LIMIT * 100
    return out


def rho_share_growth(rev: np.ndarray) -> float:
    share = ytd_share(rev)[1:]
    growth = chain_growth(rev) * 100
    return spearman(share, growth)


def cv_pct(x: np.ndarray) -> float:
    m = float(np.mean(x))
    if abs(m) < 1e-12:
        return 0.0
    return float(np.std(x, ddof=0) / m * 100)


def ar1(rng: np.random.Generator, n: int, phi: float, sigma: float) -> np.ndarray:
    x = np.zeros(n)
    x[0] = rng.normal(0, sigma)
    for t in range(1, n):
        x[t] = phi * x[t - 1] + rng.normal(0, sigma)
    return x


def scale_years(rev: np.ndarray, targets: np.ndarray) -> np.ndarray:
    out = rev.astype(float).copy()
    for y in range(3):
        sl = slice(y * 12, (y + 1) * 12)
        s = out[sl].sum()
        if s <= 0:
            out[sl] = targets[y] / 12
        else:
            out[sl] *= targets[y] / s
    return out


def enforce_usn_cap(rev: np.ndarray) -> np.ndarray:
    out = rev.copy()
    for y in range(3):
        sl = slice(y * 12, (y + 1) * 12)
        s = out[sl].sum()
        if s > USN_LIMIT:
            out[sl] *= (USN_LIMIT * 0.998) / s
    return np.maximum(out, 0.05)


def apply_braking(rev: np.ndarray, strength: float) -> np.ndarray:
    out = rev.copy()
    t = np.linspace(0, 1, 12)
    mult = 1 - strength * (t ** 2.2)
    for y in range(3):
        sl = slice(y * 12, (y + 1) * 12)
        out[sl] *= mult
    return out


def mix_with_ref(ref: np.ndarray, r_target: float, rng: np.random.Generator, noise_scale: float = 1.0) -> np.ndarray:
    z_ref = standardize(ref)
    own = standardize(ar1(rng, N, 0.35, 0.08) + rng.normal(0, 0.05, N) * noise_scale)
    r = float(np.clip(r_target, -0.95, 0.95))
    return r * z_ref + np.sqrt(max(1e-6, 1 - r * r)) * own


def z_to_positive(z: np.ndarray, seasonal: np.ndarray | None = None, vol: float = 0.10) -> np.ndarray:
    shape = 1 + vol * z
    if seasonal is not None:
        shape = seasonal * shape
    return np.maximum(shape, 0.05)


def year_targets_usn(share: float, rng: np.random.Generator, lo: float, hi: float) -> np.ndarray:
    inner_lo = lo + 0.004
    inner_hi = max(inner_lo + 0.002, hi - 0.004)
    drift = np.array([0.0, rng.uniform(-0.01, 0.015), rng.uniform(-0.01, 0.02)])
    vals = np.clip(share + drift, inner_lo, inner_hi) * USN_LIMIT
    return vals


def make_expenses(
    rev: np.ndarray,
    firm: Firm,
    rng: np.random.Generator,
    target_cv: float,
    target_r: float,
) -> np.ndarray:
    mean_exp = firm.mgmt_share * float(np.mean(rev)) * 1000
    if firm.mgmt_flat:
        noise = rng.normal(0, 0.012, N)
        return np.clip(mean_exp * (1 + noise), 1.0, None)
    z = standardize(mix_with_ref(rev, target_r, rng, noise_scale=1.2))
    exp = mean_exp * (1 + (target_cv / 100) * z)
    return np.clip(exp, mean_exp * 0.25, None)


def make_margin(firm: Firm, rng: np.random.Generator) -> np.ndarray:
    return np.clip(rng.normal(firm.margin_mean, firm.margin_std, N), 2.0, 28.0)


def make_headcount(firm: Firm, rng: np.random.Generator) -> np.ndarray:
    walk = np.cumsum(rng.integers(-1, 2, N))
    hc = firm.headcount + np.round(walk * 0.15) + rng.integers(-1, 2, N)
    if firm.regime == "УСН":
        return np.clip(hc, max(3, firm.headcount - 4), min(USN_HC_LIMIT, firm.headcount + 6)).astype(int)
    return np.clip(hc, max(105, firm.headcount - 4), firm.headcount + 8).astype(int)


def make_top_buyer(firm: Firm, rng: np.random.Generator) -> np.ndarray:
    x = firm.top_buyer + rng.normal(0, 1.6, N)
    lo, hi = (70, 95) if firm.top_buyer >= 80 else (8, 70)
    return np.clip(x, lo, hi)


def _core_ok(revs: dict[str, np.ndarray]) -> bool:
    pairs = [
        pearson(revs[CORE_NAMES[i]], revs[CORE_NAMES[j]])
        for i in range(3)
        for j in range(i + 1, 3)
    ]
    if not all(0.82 <= r <= 0.91 for r in pairs):
        return False
    for firm in POOL_CORE:
        rev = revs[firm.name]
        sh = year_shares(rev)
        if np.any(sh < 90) or np.any(sh > 98) or np.any(yearly_sums(rev) > USN_LIMIT):
            return False
        if abs(float(np.mean(chain_growth(rev)) * 100)) >= 1.5:
            return False
    return all(-0.80 <= rho_share_growth(revs[n]) <= -0.50 for n in BRAKE_NAMES)


def generate_core(rng: np.random.Generator, seasonal: np.ndarray) -> dict[str, np.ndarray]:
    best = None
    for attempt in range(MAX_TRIES):
        unique = {"Северный Ветер": 0.018, "Прима": 0.034, "Гранит": 0.038}
        unique = {k: v * rng.uniform(0.7, 1.4) for k, v in unique.items()}
        brakes = {
            "Северный Ветер": rng.uniform(0.08, 0.22),
            "Прима": rng.uniform(0.28, 0.55),
            "Гранит": rng.uniform(0.26, 0.52),
        }
        C = ar1(rng, N, 0.42, 0.045)
        revs: dict[str, np.ndarray] = {}
        for firm in POOL_CORE:
            u = rng.normal(0, unique[firm.name], N)
            raw = seasonal * (1 + C) * (1 + u)
            raw = apply_braking(raw, brakes[firm.name])
            targets = year_targets_usn(firm.target_share, rng, 0.90, 0.98)
            revs[firm.name] = enforce_usn_cap(scale_years(raw, targets))
        if _core_ok(revs):
            return revs
        pairs = [
            pearson(revs[CORE_NAMES[i]], revs[CORE_NAMES[j]])
            for i in range(3)
            for j in range(i + 1, 3)
        ]
        rhos = [rho_share_growth(revs[n]) for n in BRAKE_NAMES]
        score = sum(abs(r - 0.86) for r in pairs) + sum(abs(rho + 0.65) for rho in rhos)
        if best is None or score < best[0]:
            best = (score, {k: v.copy() for k, v in revs.items()})
    return best[1]


def generate_from_core(
    rng: np.random.Generator,
    seasonal: np.ndarray,
    core: dict[str, np.ndarray],
    firm: Firm,
    r_lo: float,
    r_hi: float,
    share_lo: float,
    share_hi: float,
    extra_ok=None,
) -> np.ndarray:
    ref = np.mean([core[n] for n in CORE_NAMES], axis=0)
    target_share = firm.target_share
    best_rev, best_pen = None, 1e9
    for i in range(MAX_TRIES):
        r_goal = rng.uniform(r_lo, r_hi)
        z = mix_with_ref(ref, r_goal, rng, noise_scale=1.3)
        raw = z_to_positive(z, seasonal=None, vol=rng.uniform(0.10, 0.18))
        share = target_share if target_share is not None else float(rng.uniform(0.47, 0.63))
        targets = year_targets_usn(share, rng, share_lo / 100, share_hi / 100)
        rev = enforce_usn_cap(scale_years(raw, targets))
        r_mean = mean_r_with_core(rev, core)
        sh = year_shares(rev)
        pen = 0.0
        if r_mean < r_lo:
            pen += (r_lo - r_mean) * 20
        if r_mean > r_hi:
            pen += (r_mean - r_hi) * 20
        pen += float(np.abs(np.clip(sh, share_lo, share_hi) - sh).sum())
        if extra_ok is not None and not extra_ok(rev):
            pen += 5
        if pen < best_pen:
            best_pen, best_rev = pen, rev
        if (
            r_lo <= r_mean <= r_hi
            and np.all((sh >= share_lo) & (sh <= share_hi))
            and np.all(yearly_sums(rev) <= USN_LIMIT)
            and (extra_ok is None or extra_ok(rev))
        ):
            return rev
    return best_rev


def generate_mirror(rng: np.random.Generator, seasonal: np.ndarray, source: np.ndarray) -> np.ndarray:
    for _ in range(MAX_TRIES):
        r_goal = rng.uniform(-0.92, -0.72)
        z = mix_with_ref(source, r_goal, rng, noise_scale=1.1)
        raw = z_to_positive(z, seasonal=None, vol=0.14)
        share = float(rng.uniform(0.80, 0.90))
        targets = year_targets_usn(share, rng, 0.80, 0.90)
        rev = enforce_usn_cap(scale_years(raw, targets))
        r = pearson(rev, source)
        sh = year_shares(rev)
        if r <= R_MIRROR_MAX and r >= -0.95 and np.all((sh >= 80) & (sh <= 90)) and np.all(yearly_sums(rev) <= USN_LIMIT):
            return rev
    z = mix_with_ref(source, -0.80, rng)
    targets = np.repeat(0.85 * USN_LIMIT, 3)
    return enforce_usn_cap(scale_years(z_to_positive(z, None, vol=0.14), targets))


def generate_osno(rng: np.random.Generator, seasonal: np.ndarray, core: dict[str, np.ndarray], firm: Firm) -> np.ndarray:
    ref = np.mean([core[n] for n in CORE_NAMES], axis=0)
    for _ in range(MAX_TRIES):
        r_goal = rng.uniform(R_IND_MIN, R_IND_MAX)
        z = mix_with_ref(ref, r_goal, rng, noise_scale=1.5)
        raw = z_to_positive(z, seasonal=None, vol=rng.uniform(0.10, 0.18))
        year_mult = np.array([1.0, 1 + firm.trend, 1 + 2 * firm.trend])
        targets = firm.base_rev * 12 * year_mult * rng.uniform(0.98, 1.05, 3)
        targets = np.maximum(targets, USN_LIMIT + 15)
        rev = scale_years(raw, targets)
        r_mean = mean_r_with_core(rev, core)
        if R_IND_MIN <= r_mean <= R_IND_MAX and np.all(yearly_sums(rev) > USN_LIMIT):
            return rev
    z = mix_with_ref(ref, 0.32, rng)
    targets = np.maximum(firm.base_rev * 12 * np.array([1.0, 1.04, 1.08]), USN_LIMIT + 20)
    return scale_years(z_to_positive(z, seasonal, vol=0.12), targets)


def assemble_firm(
    firm: Firm,
    rev: np.ndarray,
    rng: np.random.Generator,
    exp_cv: float,
    exp_r: float,
) -> pd.DataFrame:
    exp = make_expenses(rev, firm, rng, exp_cv, exp_r)
    # подгоняем V и r расходов несколькими попытками, если не flat
    if not firm.mgmt_flat:
        for cv in (exp_cv, exp_cv * 0.85, exp_cv * 0.95, exp_cv * 1.05):
            for rr in (exp_r, min(0.85, exp_r + 0.08), max(0.2, exp_r - 0.08)):
                cand = make_expenses(rev, firm, rng, cv, rr)
                if abs(cv_pct(cand) - exp_cv) <= 1.5:
                    exp = cand
                    break
            else:
                continue
            break
    return pd.DataFrame(
        {
            "Дата": DATES,
            "Компания": firm.name,
            "Режим": firm.regime,
            "Выручка_млн": np.round(rev, 3),
            "Операционная_маржа_%": np.round(make_margin(firm, rng), 1),
            "Управленческие_расходы_тыс": np.round(exp, 1),
            "Численность_чел": make_headcount(firm, rng),
            "Доля_крупнейшего_покупателя_%": np.round(make_top_buyer(firm, rng), 1),
        }
    )


def generate_all(seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    seasonal = seasonal_vector()
    core = generate_core(rng, seasonal)
    frames = []
    for firm in POOL_CORE:
        frames.append(assemble_firm(firm, core[firm.name], rng, exp_cv=12.0, exp_r=0.62))

    for firm in POOL_SAT:
        rev = generate_from_core(rng, seasonal, core, firm, R_SAT_MIN, R_SAT_MAX, 82, 88)
        frames.append(assemble_firm(firm, rev, rng, exp_cv=8.5, exp_r=0.60))

    for firm in POOL_TECH:
        rev = generate_from_core(rng, seasonal, core, firm, R_SAT_MIN, R_SAT_MAX, 86, 90)
        frames.append(assemble_firm(firm, rev, rng, exp_cv=2.0, exp_r=0.05))

    for firm in MIRROR:
        rev = generate_mirror(rng, seasonal, core[firm.mirror_of])
        frames.append(assemble_firm(firm, rev, rng, exp_cv=11.0, exp_r=0.55))

    for firm in IND_OSNO:
        rev = generate_osno(rng, seasonal, core, firm)
        frames.append(assemble_firm(firm, rev, rng, exp_cv=16.0, exp_r=0.58))

    for firm in IND_USN:
        rev = generate_from_core(rng, seasonal, core, firm, R_IND_MIN, R_IND_MAX, 40, 55)
        frames.append(assemble_firm(firm, rev, rng, exp_cv=21.0, exp_r=0.55))

    for firm in IND_BORDER:
        firm = Firm(**{**firm.__dict__, "target_share": float(rng.uniform(0.47, 0.63))})
        rev = generate_from_core(rng, seasonal, core, firm, R_BORDER_MIN, R_BORDER_MAX, 45, 65)
        frames.append(assemble_firm(firm, rev, rng, exp_cv=18.0, exp_r=0.52))

    df = pd.concat(frames, ignore_index=True)
    df["Дата"] = pd.to_datetime(df["Дата"])
    return df.sort_values(["Дата", "Компания"]).reset_index(drop=True)


def pivot_rev(df: pd.DataFrame) -> pd.DataFrame:
    p = df.pivot(index="Дата", columns="Компания", values="Выручка_млн")
    return p.loc[:, [c for c in company_order() if c in p.columns]]


def company_order() -> list[str]:
    all_firms = POOL_CORE + POOL_SAT + POOL_TECH + MIRROR + IND_OSNO + IND_USN + IND_BORDER
    return [f.name for f in all_firms]


def firm_by_name(name: str) -> Firm:
    for f in POOL_CORE + POOL_SAT + POOL_TECH + MIRROR + IND_OSNO + IND_USN + IND_BORDER:
        if f.name == name:
            return f
    raise KeyError(name)


def qa_report(df: pd.DataFrame) -> str:
    p = pivot_rev(df)
    lines = ["=== ПРОВЕРКА УСЛОВИЙ ГЕНЕРАЦИИ ===", ""]
    ok_all = True

    def flag(cond: bool, msg: str) -> None:
        nonlocal ok_all
        mark = "OK" if cond else "FAIL"
        if not cond:
            ok_all = False
        lines.append(f"[{mark}] {msg}")

    lines.append("-- Ядро: взаимные r --")
    for i, a in enumerate(CORE_NAMES):
        for b in CORE_NAMES[i + 1 :]:
            r = pearson(p[a].to_numpy(), p[b].to_numpy())
            flag(0.82 <= r <= 0.91, f"r({a}; {b}) = {r:.3f}  цель [0.82; 0.91]")

    lines.append("\n-- Ядро: доля, темп, ρ, лимит --")
    for name in CORE_NAMES:
        rev = p[name].to_numpy()
        sh = year_shares(rev)
        g = float(np.mean(chain_growth(rev)) * 100)
        rho = rho_share_growth(rev)
        flag(np.all((sh >= 90) & (sh <= 98)), f"{name}: доли {np.round(sh, 2)}  цель [90; 98]")
        flag(abs(g) < 1.5, f"{name}: средний цепной темп {g:.3f}%  цель |темп|<1.5%")
        if name in BRAKE_NAMES:
            flag(-0.80 <= rho <= -0.50, f"{name}: ρ(доля; темп) = {rho:.3f}  цель [-0.80; -0.50]")
        flag(np.all(yearly_sums(rev) <= USN_LIMIT), f"{name}: суммы {np.round(yearly_sums(rev), 2)} ≤ 450")

    groups = [
        ("Активные сателлиты", [f.name for f in POOL_SAT], R_SAT_MIN, R_SAT_MAX, 82, 88),
        ("Технические сателлиты", [f.name for f in POOL_TECH], R_SAT_MIN, R_SAT_MAX, 86, 90),
        ("Независимые УСН", [f.name for f in IND_USN], R_IND_MIN, R_IND_MAX, 40, 55),
        ("Пограничные", [f.name for f in IND_BORDER], R_BORDER_MIN, R_BORDER_MAX, 45, 65),
    ]
    for title, names, rlo, rhi, slo, shi in groups:
        lines.append(f"\n-- {title} --")
        for name in names:
            rev = p[name].to_numpy()
            r = mean_r_with_core(rev, {n: p[n].to_numpy() for n in CORE_NAMES})
            sh = year_shares(rev)
            flag(rlo <= r <= rhi, f"{name}: r_ср с ядром = {r:.3f}  цель [{rlo}; {rhi}]")
            flag(np.all((sh >= slo - 0.05) & (sh <= shi + 0.05)), f"{name}: доли {np.round(sh, 2)}  цель [{slo}; {shi}]")
            flag(np.all(yearly_sums(rev) <= USN_LIMIT), f"{name}: годовые суммы ≤ 450")

    lines.append("\n-- Зеркало --")
    r_m = pearson(p["Ветер-Юг"].to_numpy(), p["Северный Ветер"].to_numpy())
    sh_m = year_shares(p["Ветер-Юг"].to_numpy())
    flag(r_m <= R_MIRROR_MAX, f"r(Ветер-Юг; Северный Ветер) = {r_m:.3f}  цель ≤ -0.70")
    flag(np.all((sh_m >= 80) & (sh_m <= 90)), f"Ветер-Юг: доли {np.round(sh_m, 1)}  цель [80; 90]")

    lines.append("\n-- Независимые ОСНО --")
    for name in [f.name for f in IND_OSNO]:
        rev = p[name].to_numpy()
        r = mean_r_with_core(rev, {n: p[n].to_numpy() for n in CORE_NAMES})
        flag(R_IND_MIN <= r <= R_IND_MAX, f"{name}: r_ср с ядром = {r:.3f}  цель [0.25; 0.40]")
        flag(np.all(yearly_sums(rev) > USN_LIMIT), f"{name}: суммы {np.round(yearly_sums(rev), 1)} > 450")
        hc = df.loc[df["Компания"] == name, "Численность_чел"].to_numpy()
        flag(np.all(hc >= 105), f"{name}: численность min={hc.min()} ≥ 105")

    lines.append("\n-- Расходы: V и r(выручка↔расходы) --")
    for name in company_order():
        sub = df[df["Компания"] == name]
        rev = sub["Выручка_млн"].to_numpy()
        exp = sub["Управленческие_расходы_тыс"].to_numpy()
        v = cv_pct(exp)
        r_re = pearson(rev, exp)
        extra = ""
        ok = True
        if name in [f.name for f in POOL_TECH]:
            ok = v < 8 and r_re < 0.2
            extra = "цель V<8 и r<0.2"
        elif name in [f.name for f in POOL_SAT]:
            ok = 6 <= v <= 12 and 0.45 <= r_re <= 0.80
            extra = "цель V~8–9, r 0.5–0.7"
        elif name in [f.name for f in IND_BORDER]:
            ok = v > 15
            extra = "цель V>15"
        flag(ok, f"{name}: V={v:.1f}%, r(выр↔расх)={r_re:.3f}  {extra}")

    lines.append("\n-- УСН: лимит по годам --")
    usn_names = [n for n in company_order() if df.loc[df["Компания"] == n, "Режим"].iloc[0] == "УСН"]
    for name in usn_names:
        flag(np.all(yearly_sums(p[name].to_numpy()) <= USN_LIMIT), f"{name}: {np.round(yearly_sums(p[name].to_numpy()), 2)}")

    lines.append("")
    lines.append("ИТОГО: все проверки пройдены." if ok_all else "ИТОГО: есть отклонения, см. FAIL выше.")
    lines.append(f"Строк: {len(df)}, компаний: {df['Компания'].nunique()}, месяцев: {df['Дата'].nunique()}")
    return "\n".join(lines)


def save_excel(df: pd.DataFrame, path: str | Path = OUTPUT_FILE) -> None:
    path = Path(path)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name=SHEET_NAME, index=False)
        ws = writer.sheets[SHEET_NAME]
        header_font = Font(bold=True, color="FFFFFF")
        header_fill = PatternFill("solid", fgColor="305496")
        center = Alignment(horizontal="center", vertical="center")
        thin = Border(
            left=Side(style="thin", color="B0B0B0"),
            right=Side(style="thin", color="B0B0B0"),
            top=Side(style="thin", color="B0B0B0"),
            bottom=Side(style="thin", color="B0B0B0"),
        )
        widths = [14, 22, 10, 14, 22, 28, 16, 32]
        for i, width in enumerate(widths, start=1):
            cell = ws.cell(1, i)
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = center
            ws.column_dimensions[get_column_letter(i)].width = width
        for row in ws.iter_rows(min_row=2, max_row=ws.max_row, max_col=8):
            for cell in row:
                cell.border = thin
                if cell.column == 1:
                    cell.number_format = "YYYY-MM-DD"
                    cell.alignment = center
                elif cell.column in (4, 5, 6, 8):
                    cell.number_format = "0.0"
                    cell.alignment = center
                elif cell.column == 7:
                    cell.alignment = center
        ws.auto_filter.ref = ws.dimensions
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = f"A1:H{ws.max_row}"


def main() -> None:
    df = generate_all()
    save_excel(df)
    print(qa_report(df))
    print(f"\nФайл сохранён: {OUTPUT_FILE}")


main()


## Проверка варианта 1

Ячейка читает лист «Вариант 1» из ЛР6_данные.xlsx и считает показатели по уже сгенерированным рядам.

Сначала печатается словарь фирмы: годовая выручка за 2023–2025, флаг в_пределах_лимита_усн (каждый год не выше 450 млн) и средняя доля крупнейшего контрагента. Затем — проверка маркеров ядра, сателлитов, зеркала, независимых компаний и пограничных ловушек.


In [1]:
"""Проверка сгенерированных данных варианта 1 по условиям generationMD."""

from pprint import pprint

import numpy as np
import pandas as pd

USN_LIMIT = 450.0
FILE = "ЛР6_данные.xlsx"
SHEET = "Вариант 1"

CORE = ["Северный Ветер", "Прима", "Гранит"]
SAT = ["Ветер-Опт", "Ветер-Логистик"]
TECH = ["Прима-Трейд", "Гранит-Плюс"]
MIRROR = "Ветер-Юг"
MIRROR_OF = "Северный Ветер"
OSNO = ["Лагуна", "Меркурий", "Орион"]
USN_IND = ["Лагуна-Строй", "Меркурий-Юг"]
BORDER = ["Ветер-Сервис", "Прима-Снаб", "Прима-Групп"]
ORDER = CORE + SAT + TECH + [MIRROR] + OSNO + USN_IND + BORDER
BRAKE = {"Прима", "Гранит"}

df = pd.read_excel(FILE, sheet_name=SHEET)
df["Дата"] = pd.to_datetime(df["Дата"])
df["Год"] = df["Дата"].dt.year


def pearson(a, b) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def spearman(a, b) -> float:
    return pearson(pd.Series(a).rank(), pd.Series(b).rank())


def series(name: str, column: str) -> np.ndarray:
    part = df[df["Компания"] == name].sort_values("Дата")
    return part[column].to_numpy(dtype=float)


def yearly(name: str) -> dict[int, float]:
    part = df[df["Компания"] == name]
    sums = part.groupby("Год")["Выручка_млн"].sum()
    return {int(year): round(float(sums.loc[year]), 2) for year in (2023, 2024, 2025)}


def within_usn(revenue_by_year: dict[int, float]) -> bool:
    return all(value <= USN_LIMIT for value in revenue_by_year.values())


def shares(name: str) -> dict[int, float]:
    return {year: round(value / USN_LIMIT * 100, 2) for year, value in yearly(name).items()}


def mean_growth(name: str) -> float:
    rev = series(name, "Выручка_млн")
    return float(np.mean(np.diff(rev) / np.maximum(rev[:-1], 1e-9)) * 100)


def rho_share_growth(name: str) -> float:
    """Спирмен между накопленной с начала года долей лимита и цепным темпом прироста."""
    rev = series(name, "Выручка_млн")
    ytd = np.zeros_like(rev)
    for start in (0, 12, 24):
        ytd[start : start + 12] = np.cumsum(rev[start : start + 12]) / USN_LIMIT * 100
    growth = np.diff(rev) / np.maximum(rev[:-1], 1e-9) * 100
    return spearman(ytd[1:], growth)


def cv_pct(name: str) -> float:
    exp = series(name, "Управленческие_расходы_тыс")
    return float(np.std(exp, ddof=0) / np.mean(exp) * 100)


def mean_r_core(name: str) -> float:
    left = series(name, "Выручка_млн")
    return float(np.mean([pearson(left, series(other, "Выручка_млн")) for other in CORE]))


def in_band(values, lo, hi, tol=0.0) -> bool:
    return all(lo - tol <= value <= hi + tol for value in values)


фирмы = {}
for name in ORDER:
    revenue = yearly(name)
    buyer = series(name, "Доля_крупнейшего_покупателя_%")
    фирмы[name] = {
        "годовая_выручка_млн": revenue,
        "в_пределах_лимита_усн": within_usn(revenue),
        "доля_крупнейшего_контрагента_%": round(float(np.mean(buyer)), 1),
    }

print("Словарь фирм: годовая выручка, флаг лимита УСН (True — каждый год ≤ 450 млн), средняя доля крупнейшего контрагента")
print("Для компаний на ОСНО флаг False ожидаем: их выручка должна быть выше лимита УСН.")
pprint(фирмы, sort_dicts=False, width=120)

checks: list[tuple[bool, str]] = []


def check(ok: bool, text: str) -> None:
    checks.append((bool(ok), text))


print("\nПроверка условий варианта 1\n")

for i, left in enumerate(CORE):
    for right in CORE[i + 1 :]:
        r = pearson(series(left, "Выручка_млн"), series(right, "Выручка_млн"))
        check(0.82 <= r <= 0.91, f"ядро: r({left}; {right}) = {r:.3f}, цель [0.82; 0.91]")

for name in CORE:
    sh = shares(name)
    growth = mean_growth(name)
    check(in_band(sh.values(), 90, 98, tol=0.05), f"ядро: {name}, доля от лимита {sh}, цель [90; 98]%")
    check(abs(growth) < 1.5, f"ядро: {name}, средний цепной темп {growth:.3f}%, цель |темп| < 1.5%")
    check(фирмы[name]["в_пределах_лимита_усн"], f"ядро: {name}, годовая выручка ≤ 450 млн")
    if name in BRAKE:
        rho = rho_share_growth(name)
        check(-0.80 <= rho <= -0.50, f"ядро: {name}, ρ(доля; темп) = {rho:.3f}, цель [-0.80; -0.50]")

for name in SAT:
    r = mean_r_core(name)
    sh = shares(name)
    variation = cv_pct(name)
    link = pearson(series(name, "Выручка_млн"), series(name, "Управленческие_расходы_тыс"))
    check(0.55 <= r <= 0.85, f"активный сателлит: {name}, r с ядром = {r:.3f}, цель [0.55; 0.85]")
    check(in_band(sh.values(), 82, 88, tol=0.05), f"активный сателлит: {name}, доля от лимита {sh}, цель [82; 88]%")
    check(8 <= variation <= 9, f"активный сателлит: {name}, V расходов = {variation:.1f}%, цель ~8–9%")
    check(0.5 <= link <= 0.7, f"активный сателлит: {name}, r(выручка↔расходы) = {link:.3f}, цель [0.5; 0.7]")
    check(фирмы[name]["в_пределах_лимита_усн"], f"активный сателлит: {name}, годовая выручка ≤ 450 млн")

for name in TECH:
    r = mean_r_core(name)
    sh = shares(name)
    variation = cv_pct(name)
    link = pearson(series(name, "Выручка_млн"), series(name, "Управленческие_расходы_тыс"))
    check(0.55 <= r <= 0.85, f"технический сателлит: {name}, r с ядром = {r:.3f}, цель [0.55; 0.85]")
    check(variation < 8, f"технический сателлит: {name}, V расходов = {variation:.1f}%, цель < 8%")
    check(link < 0.2, f"технический сателлит: {name}, r(выручка↔расходы) = {link:.3f}, цель < 0.2")
    check(in_band(sh.values(), 86, 90, tol=0.05), f"технический сателлит: {name}, доля от лимита {sh}, цель [86; 90]%")
    check(фирмы[name]["в_пределах_лимита_усн"], f"технический сателлит: {name}, годовая выручка ≤ 450 млн")

r_mirror = pearson(series(MIRROR, "Выручка_млн"), series(MIRROR_OF, "Выручка_млн"))
sh_mirror = shares(MIRROR)
check(r_mirror <= -0.70, f"зеркало: r({MIRROR}; {MIRROR_OF}) = {r_mirror:.3f}, цель ≤ -0.70")
check(in_band(sh_mirror.values(), 80, 90, tol=0.05), f"зеркало: доля от лимита {sh_mirror}, цель [80; 90]%")
check(фирмы[MIRROR]["в_пределах_лимита_усн"], f"зеркало: {MIRROR}, годовая выручка ≤ 450 млн")

for name in OSNO:
    r = mean_r_core(name)
    revenue = фирмы[name]["годовая_выручка_млн"]
    variation = cv_pct(name)
    headcount = series(name, "Численность_чел")
    check(all(value > USN_LIMIT for value in revenue.values()), f"ОСНО: {name}, годовая выручка {revenue} > 450 млн")
    check(0.25 <= r <= 0.40, f"ОСНО: {name}, r с ядром = {r:.3f}, цель [0.25; 0.40]")
    check(15 <= variation <= 17, f"ОСНО: {name}, V расходов = {variation:.1f}%, цель ~15–17%")
    check(float(np.min(headcount)) >= 105, f"ОСНО: {name}, численность min = {int(np.min(headcount))}, цель ≥ 105")

for name in USN_IND:
    r = mean_r_core(name)
    sh = shares(name)
    variation = cv_pct(name)
    check(0.25 <= r <= 0.40, f"независимая УСН: {name}, r с ядром = {r:.3f}, цель [0.25; 0.40]")
    check(in_band(sh.values(), 40, 55, tol=0.05), f"независимая УСН: {name}, доля от лимита {sh}, цель [40; 55]%")
    check(20 <= variation <= 22, f"независимая УСН: {name}, V расходов = {variation:.1f}%, цель ~20–22%")
    check(фирмы[name]["в_пределах_лимита_усн"], f"независимая УСН: {name}, годовая выручка ≤ 450 млн")

for name in BORDER:
    r = mean_r_core(name)
    sh = shares(name)
    variation = cv_pct(name)
    check(0.20 <= r <= 0.42, f"пограничная: {name}, r с ядром = {r:.3f}, цель [0.20; 0.42]")
    check(in_band(sh.values(), 45, 65, tol=0.05), f"пограничная: {name}, доля от лимита {sh}, цель [45; 65]%")
    check(variation > 15, f"пограничная: {name}, V расходов = {variation:.1f}%, цель > 15%")
    check(фирмы[name]["в_пределах_лимита_усн"], f"пограничная: {name}, годовая выручка ≤ 450 млн")

for ok, text in checks:
    print(f"[{'OK' if ok else 'FAIL'}] {text}")

failed = [text for ok, text in checks if not ok]
print()
if failed:
    print(f"Итог: не выполнено {len(failed)} из {len(checks)}.")
else:
    print(f"Итог: все {len(checks)} условий выполнены.")


Словарь фирм: годовая выручка, флаг лимита УСН (True — каждый год ≤ 450 млн), средняя доля крупнейшего контрагента
Для компаний на ОСНО флаг False ожидаем: их выручка должна быть выше лимита УСН.
{'Северный Ветер': {'годовая_выручка_млн': {2023: 423.0, 2024: 426.65, 2025: 421.22},
                    'в_пределах_лимита_усн': True,
                    'доля_крупнейшего_контрагента_%': 35.3},
 'Прима': {'годовая_выручка_млн': {2023: 427.5, 2024: 428.59, 2025: 436.07},
           'в_пределах_лимита_усн': True,
           'доля_крупнейшего_контрагента_%': 40.3},
 'Гранит': {'годовая_выручка_млн': {2023: 423.0, 2024: 420.64, 2025: 421.2},
            'в_пределах_лимита_усн': True,
            'доля_крупнейшего_контрагента_%': 38.1},
 'Ветер-Опт': {'годовая_выручка_млн': {2023: 387.0, 2024: 383.39, 2025: 383.16},
               'в_пределах_лимита_усн': True,
               'доля_крупнейшего_контрагента_%': 45.1},
 'Ветер-Логистик': {'годовая_выручка_млн': {2023: 378.0, 2024: 378.29, 2025: 37